# 🖼️ ASL Image Processing Pipeline
### Denoise → Normalize → Augment → Convert to YOLO

| Step | Process |
|------|---------|
| 1 | Install & import |
| 2 | Configuration |
| 3 | **Denoise** — Gaussian vs Median vs Non-Local Means comparison |
| 4 | **Normalize** — pixel values to [0, 1] |
| 5 | **Augment** — Rotate, Brightness/Contrast, Flip (safe for ASL) |
| 6 | Save processed images |
| 7 | Convert to YOLO format |

---
## 📦 Step 1 — Install & Import

In [ ]:
import sys
!{sys.executable} -m pip install opencv-python numpy matplotlib pillow pyyaml tqdm scikit-image -q
print('✅ Done')

In [ ]:
import os, shutil, random, yaml, warnings
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm import tqdm
from skimage import exposure

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
print('✅ All imports OK')

---
## ⚙️ Step 2 — Configuration

In [ ]:
# ── UPDATE THESE PATHS ────────────────────────────────────────
DATASET_ROOT = '/home/abdelrahman/.cache/kagglehub/datasets/grassknoted/asl-alphabet/versions/1/asl_alphabet_train/asl_alphabet_train'
OUTPUT_ROOT  = './asl_processed'   # processed images saved here
YOLO_ROOT    = './asl_yolo'        # final YOLO dataset

# ── Sampling (same as before) ────────────────────────────────
RANGES = [
    (0,    750,  50),
    (750,  1250, 50),
    (1250, 1700, 50),
    (2000, 3000, 50),
]

# ── Denoising ───────────────────────────────────────────────
DENOISE_METHOD = 'nlm'   # 'gaussian' | 'median' | 'nlm' | chosen after comparison

# ── Augmentation per original image ─────────────────────────
AUG_PER_IMAGE = 3        # generates 3 extra augmented copies per image
                         # total per class = 200 original + 600 aug = 800

# ── Train/Val split ─────────────────────────────────────────
VAL_SPLIT = 0.15

IMG_SIZE = 200

CLASSES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z',
    'del','nothing','space'
]

print(f'📁 Source   : {DATASET_ROOT}')
print(f'📁 Output   : {OUTPUT_ROOT}')
print(f'📁 YOLO     : {YOLO_ROOT}')
print(f'🏷️  Classes  : {len(CLASSES)}')
print(f'🔁 Aug/img  : {AUG_PER_IMAGE}  → ~{200*(1+AUG_PER_IMAGE)} total/class')

---
## 🔬 Step 3 — Denoise Comparison
> Visual comparison of **Gaussian Blur vs Median Filter vs Non-Local Means** on sample images.

In [ ]:
def denoise_gaussian(img):
    """Gaussian Blur — fast, good for Gaussian noise"""
    return cv2.GaussianBlur(img, (3, 3), 0)

def denoise_median(img):
    """Median Filter — great for salt-and-pepper noise"""
    return cv2.medianBlur(img, 3)

def denoise_nlm(img):
    """Non-Local Means — best quality, preserves edges, slower"""
    return cv2.fastNlMeansDenoisingColored(
        img,
        None,
        h           = 10,   # luminance filter strength
        hColor      = 10,   # color filter strength
        templateWindowSize = 7,
        searchWindowSize   = 21,
    )

DENOISE_FNS = {
    'gaussian': denoise_gaussian,
    'median'  : denoise_median,
    'nlm'     : denoise_nlm,
}

print('✅ Denoise functions defined')

In [ ]:
# ── Pick 3 sample images from different classes ───────────
root = Path(DATASET_ROOT)
sample_imgs = []
for cls in random.sample(CLASSES[:10], 3):
    imgs = list((root / cls).glob('*.jpg'))
    if imgs:
        sample_imgs.append(random.choice(imgs))

# ── Plot comparison ───────────────────────────────────────
methods = ['Original', 'Gaussian', 'Median', 'Non-Local Means']
fig, axes = plt.subplots(len(sample_imgs), 4, figsize=(16, 4*len(sample_imgs)))
fig.suptitle('Denoising Comparison', fontsize=16, fontweight='bold', y=1.01)

for r, img_path in enumerate(sample_imgs):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    results = [
        img,
        cv2.cvtColor(denoise_gaussian(cv2.cvtColor(img, cv2.COLOR_RGB2BGR)), cv2.COLOR_BGR2RGB),
        cv2.cvtColor(denoise_median  (cv2.cvtColor(img, cv2.COLOR_RGB2BGR)), cv2.COLOR_BGR2RGB),
        cv2.cvtColor(denoise_nlm     (cv2.cvtColor(img, cv2.COLOR_RGB2BGR)), cv2.COLOR_BGR2RGB),
    ]
    for c, (res, method) in enumerate(zip(results, methods)):
        axes[r][c].imshow(res)
        axes[r][c].set_title(method, fontsize=11, fontweight='bold' if c==0 else 'normal')
        axes[r][c].axis('off')
        if c == 0:
            axes[r][c].set_ylabel(img_path.parent.name, fontsize=12, rotation=0, labelpad=40)

plt.tight_layout()
plt.savefig('denoise_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Saved → denoise_comparison.png')

In [ ]:
# ── Choose your denoising method ──────────────────────────
# Change DENOISE_METHOD to: 'gaussian' | 'median' | 'nlm'
DENOISE_METHOD = 'nlm'   # ← Non-Local Means recommended for ASL

print(f'✅ Selected denoising method: {DENOISE_METHOD.upper()}')

---
## ⚖️ Step 4 — Normalize
> Scale pixel values from `[0, 255]` → `[0, 1]` then back to `[0, 255]` uint8 for saving.
> Also applies **CLAHE** (Contrast Limited Adaptive Histogram Equalization) to enhance local contrast.

In [ ]:
def normalize_image(img_bgr):
    """
    1. Convert to LAB color space
    2. Apply CLAHE on L channel (contrast equalization)
    3. Convert back to BGR
    4. Normalize to [0,1] then rescale to uint8
    """
    # CLAHE on L channel
    lab   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img_clahe = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    # Normalize to [0,1] then back to uint8
    img_norm = img_clahe.astype(np.float32) / 255.0
    img_uint8 = (img_norm * 255).astype(np.uint8)
    return img_uint8

# ── Visual check ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Normalization (CLAHE)', fontsize=14, fontweight='bold')

test_img_path = sample_imgs[0]
orig = cv2.imread(str(test_img_path))
denoised = DENOISE_FNS[DENOISE_METHOD](orig)
normalized = normalize_image(denoised)

for ax, img, title in zip(axes,
    [orig, denoised, normalized],
    ['Original', f'Denoised ({DENOISE_METHOD.upper()})', 'Normalized (CLAHE)']):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 🔄 Step 5 — Data Augmentation
> **Safe for ASL:** Rotate ±15°, Brightness/Contrast variation
> **Flip:** only vertical flip (horizontal flip changes letter meaning in ASL!)

In [ ]:
def augment_image(img_bgr, seed=None):
    """
    Returns one randomly augmented version of the image.
    Augmentations applied:
      - Rotation       : ±15 degrees
      - Brightness     : ±30%
      - Contrast       : ±30%
      - Vertical flip  : 30% chance (safe for ASL)
    """
    if seed is not None:
        np.random.seed(seed)

    h, w = img_bgr.shape[:2]
    out  = img_bgr.copy()

    # 1. Rotation ±15°
    angle = np.random.uniform(-15, 15)
    M     = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    out   = cv2.warpAffine(out, M, (w, h),
                            borderMode=cv2.BORDER_REFLECT)

    # 2. Brightness & Contrast
    alpha = np.random.uniform(0.7, 1.3)   # contrast
    beta  = np.random.uniform(-30, 30)    # brightness
    out   = cv2.convertScaleAbs(out, alpha=alpha, beta=beta)

    # 3. Vertical flip (30% chance — safe for ASL)
    if np.random.rand() < 0.3:
        out = cv2.flip(out, 0)   # 0 = vertical flip

    return out


# ── Visual check: show original + 3 augmented versions ───
base_img = normalize_image(DENOISE_FNS[DENOISE_METHOD](cv2.imread(str(sample_imgs[0]))))

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('Augmentation Examples', fontsize=14, fontweight='bold')

axes[0].imshow(cv2.cvtColor(base_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Processed (base)', fontweight='bold')
axes[0].axis('off')

for i in range(1, 4):
    aug = augment_image(base_img, seed=i*10)
    axes[i].imshow(cv2.cvtColor(aug, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'Augmented #{i}')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('augmentation_preview.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Saved → augmentation_preview.png')

---
## 💾 Step 6 — Process & Save All Images

In [ ]:
def sample_from_ranges(images, ranges):
    images_sorted = sorted(images, key=lambda p: p.name)
    total    = len(images_sorted)
    selected = []
    for (start, end, n) in ranges:
        actual_end   = min(end, total)
        actual_start = min(start, actual_end)
        bucket       = images_sorted[actual_start:actual_end]
        selected.extend(random.sample(bucket, min(n, len(bucket))))
    return selected


def process_and_save(dataset_root, output_root, ranges,
                     denoise_method, aug_per_image, classes):

    root    = Path(dataset_root)
    out     = Path(output_root)
    denoise = DENOISE_FNS[denoise_method]

    total_saved = 0

    for cls_name in tqdm(classes, desc='Processing classes'):
        cls_dir  = root / cls_name
        save_dir = out  / cls_name
        save_dir.mkdir(parents=True, exist_ok=True)

        all_imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
        if not all_imgs:
            print(f'  ⚠️  No images in {cls_dir}')
            continue

        selected = sample_from_ranges(all_imgs, ranges)

        for img_path in selected:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            # ── Pipeline: Denoise → Normalize ──────────────
            img = denoise(img)
            img = normalize_image(img)

            # ── Save original processed image ───────────────
            stem   = img_path.stem
            cv2.imwrite(str(save_dir / f'{stem}_proc.jpg'), img)
            total_saved += 1

            # ── Save augmented copies ────────────────────────
            for a in range(aug_per_image):
                aug = augment_image(img, seed=hash(stem) + a)
                cv2.imwrite(str(save_dir / f'{stem}_aug{a}.jpg'), aug)
                total_saved += 1

    print(f'\n✅ Saved {total_saved} total images → {out.resolve()}')
    return total_saved


total = process_and_save(
    DATASET_ROOT, OUTPUT_ROOT,
    RANGES, DENOISE_METHOD,
    AUG_PER_IMAGE, CLASSES
)

---
## 🎯 Step 7 — Convert to YOLO Format

In [ ]:
def write_label(label_path, class_id):
    label_path.parent.mkdir(parents=True, exist_ok=True)
    with open(label_path, 'w') as f:
        # Full-image bbox (already cropped): x_c y_c w h all = 0.5/1.0
        f.write(f'{class_id} 0.5 0.5 1.0 1.0\n')


def convert_to_yolo(processed_root, yolo_root, classes, val_split):
    src      = Path(processed_root)
    dst      = Path(yolo_root)
    class2id = {c: i for i, c in enumerate(classes)}

    for split in ('train', 'val'):
        (dst / 'images' / split).mkdir(parents=True, exist_ok=True)
        (dst / 'labels' / split).mkdir(parents=True, exist_ok=True)

    total_train = total_val = 0

    for cls_name in tqdm(classes, desc='Converting to YOLO'):
        cls_dir = src / cls_name
        images  = list(cls_dir.glob('*.jpg'))
        if not images:
            continue

        random.shuffle(images)
        n_val  = max(1, int(len(images) * val_split))
        splits = {'val': images[:n_val], 'train': images[n_val:]}

        for split, split_imgs in splits.items():
            for img_path in split_imgs:
                dst_img = dst / 'images' / split / f'{cls_name}_{img_path.name}'
                dst_lbl = dst / 'labels' / split / f'{cls_name}_{img_path.stem}.txt'
                shutil.copy2(img_path, dst_img)
                write_label(dst_lbl, class2id[cls_name])

        total_train += len(splits['train'])
        total_val   += len(splits['val'])

    # Write data.yaml
    yaml_path = dst / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump({
            'path' : str(dst.resolve()),
            'train': 'images/train',
            'val'  : 'images/val',
            'nc'   : len(classes),
            'names': classes,
        }, f, default_flow_style=False, sort_keys=False)

    print('\n' + '='*55)
    print('  ✅ YOLO dataset ready!')
    print(f'  Train images : {total_train}')
    print(f'  Val   images : {total_val}')
    print(f'  Classes      : {len(classes)}')
    print(f'  Output       : {dst.resolve()}')
    print(f'  YAML         : {yaml_path.resolve()}')
    print('='*55)


convert_to_yolo(OUTPUT_ROOT, YOLO_ROOT, CLASSES, VAL_SPLIT)

---
## 📊 Step 8 — Final Dataset Summary

In [ ]:
yolo = Path(YOLO_ROOT)

train_imgs = list((yolo / 'images' / 'train').glob('*.jpg'))
val_imgs   = list((yolo / 'images' / 'val').glob('*.jpg'))

# Count per class
from collections import Counter
train_counts = Counter(p.name.split('_')[0] for p in train_imgs)
val_counts   = Counter(p.name.split('_')[0] for p in val_imgs)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('Final YOLO Dataset Distribution', fontsize=14, fontweight='bold')

for ax, counts, title, color in zip(
    axes,
    [train_counts, val_counts],
    ['Train Set', 'Val Set'],
    ['steelblue', 'coral']
):
    cls_names = [c for c in CLASSES if c in counts]
    cls_vals  = [counts[c] for c in cls_names]
    bars = ax.bar(cls_names, cls_vals, color=color, edgecolor='white')
    ax.set_title(f'{title} — {sum(cls_vals)} images', fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Images')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, cls_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                str(val), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n🎉 Pipeline complete!')
print(f'   Processed images : {OUTPUT_ROOT}')
print(f'   YOLO dataset     : {YOLO_ROOT}')
print(f'\n🚀 Next step → run asl_yolov8m_local.ipynb to train!')